<span style="color:green">This file scrapes the meteo for each region using the open-meteo api</span>


In [2]:
import time

import pandas as pd
import requests

url = "https://archive-api.open-meteo.com/v1/archive"

# Coordinates for the "central point" of each region
regions = {
    "Wallonia": (50.5, 4.9),
    "Flanders": (51.0, 4.5),
    "Brussels": (50.85, 4.35),
}

hourly_params = [
    "temperature_2m",
    "apparent_temperature",
    "dewpoint_2m",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "snowfall",
    "cloudcover",
    "cloudcover_low",
    "cloudcover_mid",
    "cloudcover_high",
    "windgusts_10m",
    "shortwave_radiation",
    "direct_radiation",
    "diffuse_radiation",
    "direct_normal_irradiance",
    "et0_fao_evapotranspiration",
]

for region, (latitude, longitude) in regions.items():
    print(f"Fetching data for {region}...")

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": "2014-01-01",
        "end_date": "2024-12-31",
        "hourly": hourly_params,
        "timezone": "Europe/Brussels",
    }
    # Delay needed due otherwise the requests after the first will fail due to
    # limit
    time.sleep(100)
    response = requests.get(url, params=params)

    data = response.json()

    weather_data = pd.DataFrame(data["hourly"])

    weather_data["time"] = pd.to_datetime(weather_data["time"])

    filename = f"../data/weather/{region}_hourly_weather_2014_2024.csv"
    weather_data.to_csv(filename, index=False)

    print(f"Saved {region} data to {filename}")

Fetching data for Wallonia...
Saved Wallonia data to Wallonia_hourly_weather_2014_2024.csv
Fetching data for Flanders...
Saved Flanders data to Flanders_hourly_weather_2014_2024.csv
Fetching data for Brussels...
Saved Brussels data to Brussels_hourly_weather_2014_2024.csv


In [3]:
wallonia_weather = pd.read_csv(
    "../data/weather/wallonia_hourly_weather_2014_2024.csv"
)
flanders_weather = pd.read_csv(
    "../data/weather/Flanders_hourly_weather_2014_2024.csv"
)
brussels_weather = pd.read_csv(
    "../data/weather/Brussels_hourly_weather_2014_2024.csv"
)

wallonia_weather["Region"] = 1
flanders_weather["Region"] = 2
brussels_weather["Region"] = 3

weather_data = pd.concat(
    [wallonia_weather, flanders_weather, brussels_weather]
)

weather_data["time"] = pd.to_datetime(weather_data["time"])
weather_data["date"] = weather_data["time"].dt.strftime("%Y-%m-%d")
weather_data["hour"] = weather_data["time"].dt.strftime("%H:%M:%S")
weather_data.drop(columns=["time"], inplace=True)


weather_data.to_csv(
    "../data/weather/belgium_hourly_weather_2014_2024.csv", index=False
)
print("Saved Belgium data to belgium_hourly_weather_2014_2024.csv")

Saved Belgium data to belgium_hourly_weather_2014_2024.csv
